> ⚠️ **Superseded.** These confidence intervals use Newey-West/HAC errors and are no longer the team's numbers. The corrected HC3 results are in [`regression_did_hc3_and_anticipation.ipynb`](regression_did_hc3_and_anticipation.ipynb). Kept for the record; don't cite these CIs.

# Issue #31 — Two-way FE DiD ATT and 95% CI

Uses only **United States + Canada** and reuses `src/simple_did.py` helpers. The treatment-effect coefficient is `digital_feature_available`; market and week fixed effects are included with `C(market)` and `C(week_start)`. Standard errors are Newey-West/HAC via `sd.smf_ols()`.

In [1]:
from pathlib import Path
import sys
import pandas as pd

# Works from notebooks/simple_did or notebooks/. Finds repo root by looking for src/simple_did.py.
CANDIDATE_ROOTS = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
PROJECT_ROOT = next((p for p in CANDIDATE_ROOTS if (p / "src" / "simple_did.py").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find repo root containing src/simple_did.py")

sys.path.append(str(PROJECT_ROOT / "src"))
import simple_did as sd

TREATED = "United States"
CONTROL = "Canada"
METRIC = sd.METRIC  # revenue_per_session
AUTHOR = "@gaurvsingh095"

df = sd.load_panel()
us_ca = df[df["market"].isin([TREATED, CONTROL])].copy()

print("Project root:", PROJECT_ROOT)
print("Metric:", METRIC)
print("US + Canada rows:", len(us_ca))

Project root: .
Metric: revenue_per_session
US + Canada rows: 208


In [2]:
formula = f"{METRIC} ~ digital_feature_available + C(market) + C(week_start)"
fe_model = sd.smf_ols(formula, us_ca)  # Newey-West/HAC errors, maxlags=8 by default
term = "digital_feature_available"
ci = fe_model.conf_int().loc[term]

fe_att = float(fe_model.params[term])
fe_se = float(fe_model.bse[term])
fe_p = float(fe_model.pvalues[term])
fe_ci_low = float(ci[0])
fe_ci_high = float(ci[1])

fe_result = pd.DataFrame([{
    "model": "two_way_fe_did",
    "term": term,
    "att": fe_att,
    "std_err": fe_se,
    "p_value": fe_p,
    "ci_low": fe_ci_low,
    "ci_high": fe_ci_high,
    "nobs": int(fe_model.nobs),
}])

fe_result

            model                       term  ...   ci_high  nobs
0  two_way_fe_did  digital_feature_available  ...  0.303897   208

[1 rows x 8 columns]

In [3]:
sd.save_estimate(
    method="two_way_fe_did",
    estimate=fe_att,
    issue=31,
    author=AUTHOR,
    treated=TREATED,
    control=CONTROL,
    metric=METRIC,
    window="all_pre_vs_post",
    ci_low=fe_ci_low,
    ci_high=fe_ci_high,
    notes="Two-way FE DiD using US + Canada only. Coefficient on digital_feature_available after absorbing market and week fixed effects; OLS with Newey-West/HAC errors via sd.smf_ols(maxlags=8).",
)

print(f"Two-way FE DiD ATT: {fe_att:+.4f} RPS")
print(f"95% CI: [{fe_ci_low:+.4f}, {fe_ci_high:+.4f}]")
print(f"p-value: {fe_p:.3e}")

Two-way FE DiD ATT: +0.2459 RPS
95% CI: [+0.1879, +0.3039]
p-value: 9.371e-17


## Result

Two-way FE DiD ATT = **+0.2459 RPS**. 95% CI = **[+0.1879, +0.3039]**. p = **9.371e-17**.

This satisfies issue #31: fitted on US + Canada only, reported ATT + CI, and saves one estimate row through `sd.save_estimate()`.